# 🎬 Mazinger Studio

**Dub any video into another language with AI** — powered by [Mazinger](https://pypi.org/project/mazinger/).

Paste a YouTube URL (or upload a file), pick a language and voice, then hit **Start Dubbing**.

### How to use

1. **Run Step 1** below to install dependencies (~2 min)
2. **Run Step 2** to download & prepare models (~5-10 min first time)
3. **Run Step 3** to launch the app — a public link will appear
4. **Open the link** and start dubbing!

### What you need

| Requirement | Details |
|---|---|
| **GPU runtime** *(recommended)* | For voice synthesis — *Runtime → Change runtime type → T4 GPU* |
| **OpenAI API key** *(optional)* | Only needed if you choose OpenAI as LLM provider. By default we use **Ollama** (local, free). |

### Supported languages

Chinese, English, French, German, Italian, Japanese, Korean, Portuguese, Russian, Spanish

### Voice options

- **Voice Themes** — 16 pre-defined voice styles (narrator, young, deep, warm, news, storyteller, kid, teen — male & female). No files needed!
- **Preset Voices** — Clone from pre-built HuggingFace profiles
- **Custom Voice** — Upload your own 10-30 sec audio clip
- **Auto-Clone** — Automatically clone the original speaker's voice from the source audio. No files or settings needed!

In [1]:
#@title 📦 Step 1: Install Dependencies { display-mode: "form" }

import subprocess, sys, shutil, time

# ── Ensure pip is available ──────────────────────────────────────
try:
    subprocess.check_call([sys.executable, "-m", "pip", "--version"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except subprocess.CalledProcessError:
    print("📥 Bootstrapping pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"],
                          stdout=subprocess.DEVNULL)

# ── PyTorch (cu126) — must be installed first to avoid ABI mismatch ──
print("Installing PyTorch (cu126)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "torch==2.10.0", "torchaudio==2.10.0", "torchvision==0.25.0",
                        "--index-url", "https://download.pytorch.org/whl/cu126"])

# ── Mazinger + Gradio ────────────────────────────────────────────
print("Installing Mazinger with Qwen TTS + Faster Whisper...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "mazinger[tts,transcribe-faster]", "gradio>=4.0"])

# ── Restore torchaudio (pip may have downgraded it) ─────────────
print("Ensuring torchaudio matches torch...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "torchaudio==2.10.0",
                        "--index-url", "https://download.pytorch.org/whl/cu126"])

Installing PyTorch (cu126)...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
whisperx 3.8.4 requires torch~=2.8.0, but you have torch 2.10.0+cu126 which is incompatible.
whisperx 3.8.4 requires torchaudio~=2.8.0, but you have torchaudio 2.10.0+cu126 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


Installing Mazinger with Qwen TTS + Faster Whisper...



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


Ensuring torchaudio matches torch...



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


0

In [2]:
#@title 📥 Step 2: Download & Prepare Models { display-mode: "form" }

#@markdown Downloads all required models to disk so they're ready when needed.
#@markdown This avoids long waits during your first dubbing run.
#@markdown
#@markdown **Expected download sizes:**
#@markdown - Ollama LLM model: ~1.5 GB
#@markdown - Faster-Whisper (large-v3): ~3 GB
#@markdown - Qwen TTS: ~3-4 GB

import subprocess, sys, shutil, time, os
import base64, io

# ── Configuration ────────────────────────────────────────────────
OLLAMA_MODEL = "qwen3.5:2b-bf16"       # ← change this to use a different model
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL  # propagate to Gradio app

# ── Helper: wait for Ollama server readiness ─────────────────────
def _wait_for_ollama(url="http://localhost:11434/api/tags", retries=15, delay=2):
    """Poll Ollama until it responds (up to retries * delay seconds)."""
    import urllib.request, urllib.error
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return True
        except (urllib.error.URLError, OSError):
            pass
        time.sleep(delay)
    return False

# ── Ollama (local LLM server) ───────────────────────────────────
if not shutil.which("ollama"):
    print("\n📥 Installing Ollama (local LLM server)...")
    subprocess.check_call(["sudo", "apt-get", "install", "-y", "-qq", "zstd"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sudo bash"],
    )
    print("✅ Ollama installed")
else:
    print("\n✅ Ollama already installed")

# Start Ollama server in background and wait until it's ready
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("⏳ Waiting for Ollama server to be ready...")
if _wait_for_ollama():
    print("✅ Ollama server is ready")
else:
    print("⚠️  Ollama server did not become ready in time — pull may fail")

# ── GPU check ────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU detected — voice synthesis will be slow.")
        print("   Tip: Runtime → Change runtime type → T4 GPU")
except ImportError:
    pass

print("\n📥 Downloading models (this may take 5-10 min on first run)...\n")

# ── 1. Ollama LLM model ─────────────────────────────────────────
print(f"1️⃣  Pulling Ollama LLM model ({OLLAMA_MODEL})...")
_ollama_ok = False
for _attempt in range(3):
    result = subprocess.run(
        ["ollama", "pull", OLLAMA_MODEL],
        timeout=600,
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        _ollama_ok = True
        print("   ✅ Ollama model ready\n")
        break
    else:
        _err = (result.stderr or result.stdout or "").strip()
        print(f"   ⚠️  Pull attempt {_attempt + 1}/3 failed: {_err}")
        time.sleep(3)

if not _ollama_ok:
    print("   ❌ Ollama model pull failed after 3 attempts.")
    print(f"      Try running manually: !ollama pull {OLLAMA_MODEL}\n")

# ── 2. Faster-Whisper model ─────────────────────────────────────
print("2️⃣  Downloading Faster-Whisper model (large-v3)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Systran/faster-whisper-large-v3")
    print("   ✅ Faster-Whisper model ready\n")
except Exception as e:
    print(f"   ⚠️  Faster-Whisper download failed: {e}\n")

# ── 3. Qwen TTS model ───────────────────────────────────────────
print("3️⃣  Downloading Qwen TTS model...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
    print("   ✅ Qwen TTS model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen TTS download failed: {e}\n")

# ── 4. Qwen VoiceDesign model (for voice themes) ────────────────
print("4️⃣  Downloading Qwen VoiceDesign model (for voice themes)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
    print("   ✅ Qwen VoiceDesign model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen VoiceDesign download failed: {e}\n")

# ── 5. Warm up Ollama (load model into GPU) ──────────────────────
print("5️⃣  Warming up Ollama LLM (loading model into GPU)...")
if _ollama_ok:
    try:
        import json, urllib.request
        _body = json.dumps({
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": "Reply with: ready"}],
            "stream": False,
            "think": False,
            "options": {"temperature": 0.1},
        }).encode()
        _req = urllib.request.Request(
            "http://localhost:11434/api/chat", _body,
            headers={"Content-Type": "application/json"},
        )
        t0 = time.time()
        with urllib.request.urlopen(_req, timeout=120) as _resp:
            _result = json.loads(_resp.read())
        _reply = _result.get("message", {}).get("content", "")
        _tokens = _result.get("eval_count", 0)
        print(f"   ✅ Ollama responded in {time.time() - t0:.1f}s ({_tokens} tokens): {_reply[:50]}\n")
    except Exception as e:
        print(f"   ⚠️  Warm-up failed: {e}\n")
else:
    print("   ⏭️  Skipped (model not available)\n")

print("✅ All models downloaded and cached! Run the next cell to launch the app.")

# ── Benchmark Ollama LLM with test image ─────────────────────────
if _ollama_ok:
    try:
        from PIL import Image as _PILImage
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow"])
        from PIL import Image as _PILImage

    _img = _PILImage.new("RGB", (64, 64), color=(200, 60, 60))
    _buf = io.BytesIO()
    _img.save(_buf, format="JPEG", quality=70)
    _b64 = base64.b64encode(_buf.getvalue()).decode()

    # ── Build mazinger LLM client (native Ollama, thinking disabled) ─
    from mazinger.llm import build_client

    _client = build_client(
        base_url="http://localhost:11434/v1",
        think=False,
    )

    _model = OLLAMA_MODEL
    _messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in one sentence."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{_b64}"}},
            ],
        }
    ]

    # ── Run benchmark ────────────────────────────────────────────────
    print(f"🏎️  Benchmarking {_model} with a 64×64 test image…\n")

    try:
        _t0 = time.time()
        _resp = _client.chat.completions.create(model=_model, messages=_messages, temperature=0.1)
        _elapsed = time.time() - _t0

        _text = _resp.choices[0].message.content
        _prompt_tok = _resp.usage.prompt_tokens
        _gen_tok = _resp.usage.completion_tokens
        _tok_per_sec = _gen_tok / _elapsed if _elapsed > 0 else 0

        print(f"   Response   : {_text[:120]}")
        print(f"   Prompt tok : {_prompt_tok}")
        print(f"   Gen tokens : {_gen_tok}")
        print(f"   Wall time  : {_elapsed:.2f}s")
        print(f"   Speed      : {_tok_per_sec:.1f} tok/s")
        print()

        if _tok_per_sec < 5:
            print("⚠️  Throughput is low — consider a smaller model or check GPU availability.")
        elif _tok_per_sec < 20:
            print("✅ Reasonable speed. Dubbing will work but may be a bit slow on long videos.")
        else:
            print("🚀 Great throughput! You're ready to dub.")
    except Exception as e:
        print(f"   ⚠️  Benchmark failed: {e}")
        print("   The model may not support vision. Dubbing will still work for text-only tasks.")
else:
    print("\n⏭️  Skipping benchmark (Ollama model not available).")
    print(f"   Try pulling the model manually: !ollama pull {OLLAMA_MODEL}")


✅ Ollama already installed
⏳ Waiting for Ollama server to be ready...
✅ Ollama server is ready

✅ GPU detected: NVIDIA GeForce RTX 4060 Ti

📥 Downloading models (this may take 5-10 min on first run)...

1️⃣  Pulling Ollama LLM model (qwen3.5:2b-bf16)...
   ⚠️  Pull attempt 1/3 failed: pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
Error: pull model manifest: 412: 
The model you are attempting to pull requires a newer version of Ollama that may be in pre-release.

Please see https://github.com/ollama/ollama/releases for more details.
   ⚠️  Pull attempt 2/3 failed: pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
Error: pull model manifest: 412: 
The model you are attempting to pull requires a newer version of Ollama that may be in pre-release.

Please see https://github.com/ollama/ollama/releases for more details.
   ⚠️  Pull attempt 3/3 failed: pulling man

/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:05<00:00,  1.31it/s]


   ✅ Faster-Whisper model ready

3️⃣  Downloading Qwen TTS model...


Fetching 13 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:05<00:00,  2.35it/s]


   ✅ Qwen TTS model ready

4️⃣  Downloading Qwen VoiceDesign model (for voice themes)...


Fetching 13 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:05<00:00,  2.36it/s]

   ✅ Qwen VoiceDesign model ready

5️⃣  Warming up Ollama LLM (loading model into GPU)...
   ⏭️  Skipped (model not available)

✅ All models downloaded and cached! Run the next cell to launch the app.

⏭️  Skipping benchmark (Ollama model not available).
   Try pulling the model manually: !ollama pull qwen3.5:2b-bf16


In [ ]:
#@title 🚀 Step 3: Launch Mazinger Studio { display-mode: "form" }

import subprocess, os, sys

# ── Ensure OLLAMA_MODEL env var is set (in case Step 2 was skipped) ──
os.environ.setdefault("OLLAMA_MODEL", "qwen3.5:2b-bf16")

# ── Fetch studio scripts from GitHub ─────────────────────────────
_BASE = "https://raw.githubusercontent.com/bakrianoo/mazinger/refs/heads/master/docs/notebooks/studio"
_SCRIPTS = ["constants.py", "theme.py", "helpers.py", "pipeline.py", "app.py"]

os.makedirs("studio", exist_ok=True)
for _script in _SCRIPTS:
    _dest = os.path.join("studio", _script)
    if not os.path.exists(_dest):
        subprocess.check_call(["wget", "-q", f"{_BASE}/{_script}", "-O", _dest])
        print(f"  ✅ Downloaded {_script}")
    else:
        print(f"  ✔ {_script} (cached)")

print(f"\n🚀 Launching Mazinger Studio (Ollama model: {os.environ['OLLAMA_MODEL']})...\n")

# ── Run the app inline so Gradio can display its link in the notebook ─
sys.path.insert(0, "studio")
from app import app
app.launch(share=True, debug=True, show_error=True)


  ✔ constants.py (cached)
  ✔ theme.py (cached)
  ✔ helpers.py (cached)
  ✔ pipeline.py (cached)
  ✔ app.py (cached)

🚀 Launching Mazinger Studio (Ollama model: qwen3.5:2b-bf16)...



/workspace/mazinger/docs/notebooks/studio/app.py:18: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Mazinger Studio", css=CSS) as app:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9f92b1746486cd098e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[youtube] Extracting URL: https://youtube.com/shorts/GJwW7LgT2I0?si=q806Y7o_utpJ1gx2
[youtube] GJwW7LgT2I0: Downloading webpage
[youtube] GJwW7LgT2I0: Downloading tv downgraded player API JSON
[youtube] GJwW7LgT2I0: Downloading web creator client config
[youtube] GJwW7LgT2I0: Downloading player 8e54e4ea-main
[youtube] GJwW7LgT2I0: Downloading web creator player API JSON
[youtube] [jsc:node] Solving JS challenges using node
[info] GJwW7LgT2I0: Downloading 1 format(s): 396+251
[download] Destination: ./mazinger_output/projects/ليه-تفطر-فول-وطعمية-لما-ممكن-تفطر-سمك-مشوي-بـ60-جنيه/source/video.f396.mp4
[download] 100% of    3.38MiB in 00:00:00 at 38.48MiB/s  
[download] Destination: ./mazinger_output/projects/ليه-تفطر-فول-وطعمية-لما-ممكن-تفطر-سمك-مشوي-بـ60-جنيه/source/video.f251.webm
[download] 100% of    1.07MiB in 00:00:00 at 25.96MiB/s  
[Merger] Merging formats into "./mazinger_output/projects/ليه-تفطر-فول-وطعمية-لما-ممكن-تفطر-سمك-مشوي-بـ60-جنيه/source/video.mp4"
Deleting original file

Translating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:43<00:00, 21.91s/it]



********
********
 
